# Forecasting / GRN Analysis

Runs forecasting inference, DEG-delta panel, TF-marker network, and optional GRN bootstrap/permutation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import scanpy as sc

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.utils.config import load_yaml_config
from model.utils.constants import get_pv_identity_markers, get_pv_path_nodes
from model.models import Forecaster, ForecasterConfig
from model.training.checkpointing import load_checkpoint
from model.data.trajectory_pairs import PrepareTrajectoryData, TrajectoryDataset
from model.data.preprocessing import prepare_clusters
from model.analysis import (
    compute_transition_deg_delta_panel,
    get_forecaster_predictions,
    extract_tf_marker_network,
    run_heatmap_pipeline,
    load_tf_set,
    GRNEvaluator,
    GRNBootstrapper,
    GRNPermutationTester,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


In [ ]:
# ===== Parameters =====
SMOKE_MODE = True            # False -> fuller reproduction
RUN_ENRICHMENT = True        # requires gseapy + Enrichr access
RUN_BOOTSTRAP = False        # expensive
RUN_PERMUTATION = False      # very expensive

SEED = int(CFG['seed']) if 'CFG' in globals() and 'seed' in CFG else 42
MAX_ITEMS = 200 if SMOKE_MODE else None
BATCH_SIZE = 32 if SMOKE_MODE else 128
ATTN_THRESHOLD = 0.005
ENRICHMENT_PADJ = 0.05

OUTDIR = ROOT / 'artifacts' / 'notebooks' / 'forecasting'
OUTDIR.mkdir(parents=True, exist_ok=True)

CFG = load_yaml_config(ROOT / 'configs' / 'forecasting.yaml')
SEED = int(CFG.get('seed', SEED))
adata_path = ROOT / CFG['data']['h5ad_path']
ckpt_path = ROOT / CFG['training']['checkpoint_path']
tf_file = ROOT / 'data' / 'Homo_sapiens_TF.html'

print('adata_path:', adata_path)
print('ckpt_path:', ckpt_path)
print('RUN_ENRICHMENT:', RUN_ENRICHMENT)


In [ ]:
adata = sc.read_h5ad(adata_path)
adata = prepare_clusters(adata, CFG['data'].get('cluster_col'))
print(adata)

prep = PrepareTrajectoryData(
    h5ad_path=str(adata_path),
    config={'model_params': CFG['model']},
    subset_col=CFG['data'].get('subset_col', 'trajectory_class'),
    subset_values=tuple(CFG['data'].get('subset_values', ['PV'])),
    time_col=CFG['data'].get('time_col'),
    cluster_col=CFG['data'].get('cluster_col'),
    n_bins=int(CFG['data'].get('n_bins', 120)),
    allowed_offsets=tuple(CFG['data'].get('allowed_offsets', [1, 2, 3, 4, 5, 6])),
    base_max_dist=float(CFG['data'].get('base_max_dist', 12.0)),
    dist_alpha=float(CFG['data'].get('dist_alpha', 1.0)),
    allowed_cross_steps=tuple(CFG['data'].get('allowed_cross_steps', [1, 2])),
    k_intra=int(CFG['data'].get('k_intra', 1)),
    k_cross=int(CFG['data'].get('k_cross', 2)),
    val_split=float(CFG['data'].get('val_split', 0.2)),
    heldout_split=float(CFG['data'].get('heldout_split', 0.1)),
    pair_diagnostics=True,
)
if len(prep.heldout_data) == 0:
    raise ValueError('Held-out dataset is empty. Increase data size or reduce heldout_split.')
if 'time_bin' not in prep.adata.obs.columns:
    raise KeyError("PrepareTrajectoryData must populate prep.adata.obs['time_bin'] for reproducible diagnostics.")

if MAX_ITEMS is None:
    eval_indices = np.arange(len(prep.heldout_data))
else:
    rng = np.random.default_rng(SEED)
    eval_indices = np.sort(rng.choice(len(prep.heldout_data), size=min(MAX_ITEMS, len(prep.heldout_data)), replace=False))

eval_data_list = [prep.heldout_data[int(i)] for i in eval_indices]
heldout_ds = TrajectoryDataset(eval_data_list)
heldout_loader = DataLoader(heldout_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'heldout total: {len(prep.heldout_data)} | evaluating: {len(heldout_ds)}')


In [ ]:
f_model = Forecaster(ForecasterConfig.from_dict(CFG['model'])).to(DEVICE)
f_model, f_ckpt = load_checkpoint(f_model, ckpt_path, device=DEVICE)
f_model.eval()
print('forecasting model loaded. checkpoint keys:', list(f_ckpt.keys())[:8])

In [ ]:
# Inference on the same held-out pairs used for attention analysis
preds, trues, nz_masks, meta = get_forecaster_predictions(f_model, heldout_loader, DEVICE)
print('preds shape:', preds.shape, 'trues shape:', trues.shape)


## 1) DEG Delta Panel

In [ ]:
deg_df = compute_transition_deg_delta_panel(
    preds=preds,
    trues=trues,
    meta=meta,
    adata=prep.adata,
    trajectory_nodes=get_pv_path_nodes(),
    cluster_col=CFG['data'].get('cluster_col'),
    top_n=50,
)
display(deg_df.head(20))
deg_df.to_csv(OUTDIR / 'deg_delta_panel.csv', index=False)
print('Saved:', OUTDIR / 'deg_delta_panel.csv')


## 2) Cross-Attention TF-Marker Network

The GRN story starts from the decoder cross-attention tensor: marker queries attend over source-cell input genes. We first extract raw TF->marker edges, then use the same edge table for significance filtering, enrichment, and optional null-model checks.


In [ ]:
known_tfs = load_tf_set(str(tf_file))
markers = get_pv_identity_markers()

print(f'Known TFs loaded: {len(known_tfs)}')
print(f'Markers requested: {markers}')

net_df = extract_tf_marker_network(
    model=f_model,
    dataset=heldout_ds,
    target_markers=markers,
    known_tfs=known_tfs,
    adata=prep.adata,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    threshold=ATTN_THRESHOLD,
    verbose=True,
)
if net_df.empty:
    raise ValueError('No TF-marker cross-attention edges were extracted. Lower ATTN_THRESHOLD or inspect marker/TF gene coverage.')

display(net_df.head(20))
net_path = OUTDIR / 'tf_marker_cross_attention_edges.csv'
net_df.to_csv(net_path, index=False)
print('Saved raw cross-attention edges:', net_path)

heat_df = run_heatmap_pipeline(net_df, top_n=30)
heat_path = OUTDIR / 'tf_marker_heatmap_table.csv'
heat_df.to_csv(heat_path, index=False)
print('Saved heatmap table:', heat_path)


## 3) Marker-Specific TF Filtering and Enrichment


In [ ]:
# Marker-wise MAD filtering followed by Reactome enrichment of selected TFs
edge_input_path = OUTDIR / 'tf_marker_cross_attention_edges.csv'
grne = GRNEvaluator(
    csv_path=str(edge_input_path),
    k_multiplier=3.5,
    target_markers=markers,
)
raw_edges = grne.load_data()
filtered_tf_summary = grne.process_data()

selected_rows = []
for marker, info in filtered_tf_summary.items():
    selected_rows.append({
        'Marker': marker,
        'threshold': info['threshold'],
        'n_total_tfs': info['n_total_tfs'],
        'n_selected_tfs': info['n_selected_tfs'],
        'selected_tfs': ';'.join(info['selected_tfs']),
    })
selected_tf_df = pd.DataFrame(selected_rows)
selected_tf_df.to_csv(OUTDIR / 'marker_selected_tfs_mad.csv', index=False)
display(selected_tf_df)

if RUN_ENRICHMENT:
    try:
        enrichment_summary = grne.run_enrichment(adjusted_p_cutoff=ENRICHMENT_PADJ)
        enrichment_df = pd.concat(
            [df for df in enrichment_summary.values() if df is not None and not df.empty],
            ignore_index=True,
        ) if any(df is not None and not df.empty for df in enrichment_summary.values()) else pd.DataFrame()
        enrichment_df.to_csv(OUTDIR / 'marker_tf_reactome_enrichment.csv', index=False)
        core_pathway_df = grne.extract_core_pathways(top_n_per_marker=10)
        core_pathway_df.to_csv(OUTDIR / 'marker_tf_core_pathways.csv', index=False)
        print('Saved enrichment tables to:', OUTDIR)
        display(core_pathway_df.head(20))
    except Exception as exc:
        enrichment_df = pd.DataFrame()
        core_pathway_df = pd.DataFrame()
        print('Enrichment skipped/failed:', exc)
else:
    enrichment_df = pd.DataFrame()
    core_pathway_df = pd.DataFrame()
    print('Enrichment skipped by RUN_ENRICHMENT=False.')


## 4) GRN Bootstrap / Permutation Nulls


In [ ]:
boot_df = pd.DataFrame()
perm_df = pd.DataFrame()

if RUN_BOOTSTRAP:
    boot = GRNBootstrapper(
        model=f_model,
        dataset=heldout_ds,
        markers=markers,
        tfs=known_tfs,
        adata=prep.adata,
        k_mad=3.0,
    )
    boot_df = boot.run_bootstrap(
        n_iterations=5 if SMOKE_MODE else 20,
        sample_frac=0.8,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        threshold=ATTN_THRESHOLD,
        verbose=True,
    )
    boot_df.to_csv(OUTDIR / 'grn_bootstrap_summary.csv', index=False)
    display(boot_df.head(20))

if RUN_PERMUTATION:
    real_df = boot_df.copy() if not boot_df.empty else net_df.rename(columns={'Score': 'Mean_Score'})
    if real_df.empty:
        print('Permutation skipped: no real cross-attention edges available.')
    else:
        perm = GRNPermutationTester(
            model=f_model,
            dataset=heldout_ds,
            markers=markers,
            tfs=known_tfs,
            adata=prep.adata,
            real_results_df=real_df,
        )
        perm_df = perm.run_permutation_test(
            n_permutations=5 if SMOKE_MODE else 20,
            device=DEVICE,
            batch_size=BATCH_SIZE,
            threshold=ATTN_THRESHOLD,
            verbose=True,
        )
        perm_df.to_csv(OUTDIR / 'grn_permutation_results.csv', index=False)
        display(perm_df.head(20))

if (not RUN_BOOTSTRAP) and (not RUN_PERMUTATION):
    print('Optional GRN robustness steps skipped.')
